# Titanic EDA

Exploratory analysis of `train.csv` only (no leakage from Kaggle's `test.csv`). This drives the feature engineering and preprocessing choices used in `src/preprocessing.py` and `train.py`.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_utils import fetch_titanic_data, load_titanic_csv

sns.set_theme(style='whitegrid')

train_path = fetch_titanic_data('../data')
df = load_titanic_csv(train_path)
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

## Missing values

`Age` has substantial missingness, `Cabin` is mostly missing, and `Embarked` has a couple of gaps. This informs the imputation strategy: median `Age` per (Title, Pclass) group, mode for `Embarked`, and treating `Cabin` as a categorical "deck" feature with an explicit "Unknown" bucket rather than dropping it.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({'missing': missing, 'pct': missing_pct})[missing > 0]

## Target balance

In [ ]:
survival_rate = df['Survived'].mean()
print(f'Overall survival rate: {survival_rate:.2%}')
ax = df['Survived'].value_counts().sort_index().plot(kind='bar')
ax.set_xticklabels(['Died (0)', 'Survived (1)'], rotation=0)
ax.set_ylabel('Count')
ax.set_title('Class balance')
plt.show()

Mild class imbalance (~62% died vs ~38% survived) — not severe enough to need resampling, but `train.py` still applies a `pos_weight` in the loss to be safe.

## Survival by sex and class

These are historically the two strongest predictors (women and children first; first-class passengers had better access to lifeboats).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=df, x='Sex', y='Survived', ax=axes[0])
axes[0].set_title('Survival rate by sex')
sns.barplot(data=df, x='Pclass', y='Survived', ax=axes[1])
axes[1].set_title('Survival rate by passenger class')
plt.tight_layout()
plt.show()

In [ ]:
sns.catplot(data=df, x='Pclass', y='Survived', hue='Sex', kind='bar', height=4, aspect=1.3)
plt.title('Survival rate by class and sex')
plt.show()

## Age distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=df, x='Age', hue='Survived', multiple='stack', bins=30, ax=ax)
ax.set_title('Age distribution by survival')
plt.show()

Young children show a noticeably higher survival rate, supporting the value of an Age-aware model rather than dropping the feature.

## Fare distribution

Highly right-skewed with outliers (some fares > 500), which is why `Fare` is standardized rather than left raw.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['Fare'], bins=40, ax=axes[0])
axes[0].set_title('Fare distribution (raw)')
sns.boxplot(data=df, x='Survived', y='Fare', ax=axes[1])
axes[1].set_title('Fare by survival')
plt.tight_layout()
plt.show()

## Family size and travelling alone

Engineered as `FamilySize = SibSp + Parch + 1` and `IsAlone`. Passengers alone or in very large families tended to survive less often than those in small families.

In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
sns.barplot(data=df, x='FamilySize', y='Survived')
plt.title('Survival rate by family size')
plt.show()

## Title extracted from Name

`Name` itself is not usable directly, but the honorific title (Mr, Mrs, Miss, Master, ...) is a strong, low-cardinality proxy for age/sex/social status and is a classic Titanic feature-engineering trick.

In [ ]:
import re
df['Title'] = df['Name'].apply(lambda n: re.search(r',\s*([^.]+)\.', n).group(1).strip())
title_survival = df.groupby('Title')['Survived'].agg(['mean', 'count']).sort_values('count', ascending=False)
title_survival

Rare titles are bucketed into a single "Rare" category in `src/preprocessing.py` to avoid extremely sparse one-hot columns.

## Embarked port

In [ ]:
sns.barplot(data=df, x='Embarked', y='Survived')
plt.title('Survival rate by embarkation port')
plt.show()

## Correlation among numeric features

In [ ]:
numeric_cols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize']
corr = df[numeric_cols].corr()
plt.figure(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation matrix')
plt.show()

## Summary of preprocessing decisions driven by this EDA

- **Age**: impute missing values with the median per (Title, Pclass) group rather than a single global median, since age varies meaningfully across those groups.
- **Fare**: impute the rare missing value with the median; standardize due to skew/outliers.
- **Embarked**: impute with the mode (only 2 missing rows).
- **Cabin**: too sparse to use directly; reduced to a `Deck` categorical feature (first letter, with an "Unknown" bucket) rather than dropped outright, since cabin/deck correlates with Pclass and location on the ship.
- **Name/Ticket/PassengerId**: dropped as identifiers, except for the `Title` extracted from `Name`.
- **FamilySize / IsAlone**: engineered from `SibSp`/`Parch` since family size has a non-monotonic relationship with survival that a single linear coefficient wouldn't capture well.
- **Categorical encoding**: one-hot encode `Pclass`, `Sex`, `Embarked`, `Title`, `Deck`.
- **Scaling**: standardize numeric features (`Age`, `Fare`, `FamilySize`, `SibSp`, `Parch`) since the model is a neural network sensitive to feature scale.